In [10]:
# Install libraries (if not already installed)
!pip install pandas scikit-learn gradio

# Import
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import gradio as gr

# Load data with correct encoding
try:
    # Try common encodings for spam datasets
    data = pd.read_csv('spam.csv', encoding='latin-1')  # Most common for this dataset
except:
    try:
        data = pd.read_csv('spam.csv', encoding='ISO-8859-1')
    except:
        data = pd.read_csv('spam.csv', encoding='utf-8', errors='ignore')

# Select and rename columns
data = data[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})

# Train model
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(data["text"])
model = MultinomialNB().fit(X_vec, data["label"])


# Prediction function
def predict(text):
    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

# Test
print(predict("Win a free prize now!"))  # Output should be "spam"

# Gradio UI
demo = gr.Interface(fn=predict, inputs="text", outputs="label")
demo.launch()

spam
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://64dd6494bd1cc35f36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
# spam_detection.py
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import gradio as gr
import joblib
import os
from urllib.request import urlretrieve

# 1. Load Data with Fallback
def load_data():
    try:
        # Try loading from GitHub
        url = "https://raw.githubusercontent.com/justmarkham/pydata-dc-2016-tutorial/master/sms.tsv"
        data = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])
        return data
    except:
        # Fallback to local file
        try:
            data = pd.read_csv('sms.csv', encoding='latin-1')
            return data[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
        except:
            # Ultimate fallback with hardcoded data
            print("Warning: Using minimal sample data")
            return pd.DataFrame({
                'text': ["Free prize now", "Hey how are you"],
                'label': ["spam", "ham"]
            })

# 2. Train and Save Model
def train_model():
    data = load_data()
    X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2)

    vectorizer = CountVectorizer()
    X_train_vec = vectorizer.fit_transform(X_train)

    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    # Save artifacts
    joblib.dump(model, 'spam_model.joblib')
    joblib.dump(vectorizer, 'vectorizer.joblib')
    return model, vectorizer

# 3. Prediction Function
def predict_spam(text):
    try:
        model = joblib.load('spam_model.joblib')
        vectorizer = joblib.load('vectorizer.joblib')
    except:
        model, vectorizer = train_model()

    text_vec = vectorizer.transform([text])
    return model.predict(text_vec)[0]

# 4. Gradio Interface
def create_interface():
    iface = gr.Interface(
        fn=predict_spam,
        inputs=gr.Textbox(lines=2, placeholder="Enter message..."),
        outputs="label",
        examples=[
            ["Hey, can we meet tomorrow?"],
            ["WINNER! Claim your $1000 prize now!"],
            ["Your package will arrive today"]
        ],
        title="📧 Spam Detection",
        description="Classifies messages as SPAM or HAM (legitimate)"
    )
    return iface

# Run the app
if __name__ == "__main__":
    create_interface().launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bb2778b6b1653fdf34.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
import gradio as gr
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import joblib

def create_interface():
    # Custom CSS for styling (now targets specific elements)
    css = """
    .gradio-container {
        font-family: 'Segoe UI', sans-serif;
        max-width: 800px;
        margin: auto;
    }
    .title {
        text-align: center;
        color: #2b5876;
        font-size: 28px;
        font-weight: 600;
        margin-bottom: 20px;
    }
    .desc-box {
        background: #f5f7fa;
        padding: 15px;
        border-radius: 8px;
        margin-bottom: 20px;
        border-left: 4px solid #2b5876;
    }
    #examples-container {
        background: #f0f4f8;
        padding: 10px;
        border-radius: 8px;
    }
    """

    # Model loading/training
    try:
        model = joblib.load('spam_model.joblib')
        vectorizer = joblib.load('vectorizer.joblib')
    except:
        data = pd.read_csv('spam.csv', encoding='latin-1')
        data = data[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
        vectorizer = CountVectorizer()
        X_vec = vectorizer.fit_transform(data["text"])
        model = MultinomialNB().fit(X_vec, data["label"])
        joblib.dump(model, 'spam_model.joblib')
        joblib.dump(vectorizer, 'vectorizer.joblib')

    def predict(text):
        text_vec = vectorizer.transform([text])
        proba = model.predict_proba(text_vec)[0]
        return {
            "spam": float(proba[1]),
            "ham": float(proba[0])
        }, pd.DataFrame({
            "class": ["ham", "spam"],
            "probability": [proba[0], proba[1]]
        })

    with gr.Blocks(css=css) as demo:
        gr.Markdown("""
        <div class='title'>📧 Advanced Spam Detection</div>
        <div class='desc-box'>
        This model classifies SMS messages as <strong>spam</strong> or <strong>ham</strong> (legitimate)
        using a Naive Bayes classifier trained on 5,574 messages.
        </div>
        """)

        with gr.Row():
            with gr.Column():
                input_text = gr.Textbox(
                    label="Enter your message",
                    placeholder="Type or paste message here...",
                    lines=4
                )
                submit_btn = gr.Button("Analyze", variant="primary")

                with gr.Accordion("Try these examples:", open=False, elem_id="examples-container"):
                    gr.Examples(
                        examples=[
                            ["WINNER! Claim your $1000 prize now!"],
                            ["Hey, can we meet tomorrow at 5pm?"],
                            ["Urgent: Your account will be suspended"],
                            ["Mom sent you some photos"]
                        ],
                        inputs=input_text,
                        label=""
                    )

            with gr.Column():
                label_output = gr.Label(
                    label="Prediction",
                    num_top_classes=2
                )
                proba_output = gr.BarPlot(
                    label="Confidence Scores",
                    x="class",
                    y="probability",
                    color="class",
                    height=300,
                    width=200,
                    container=False
                )

        submit_btn.click(
            fn=predict,
            inputs=input_text,
            outputs=[label_output, proba_output]
        )

    return demo

if __name__ == "__main__":
    create_interface().launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60a14a78576d107c02.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
import gradio as gr
import pandas as pd
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# ====================== 🌟 MODEL LOADING ======================
def load_model():
    try:
        model = joblib.load('spam_model.joblib')
        vectorizer = joblib.load('vectorizer.joblib')
        return model, vectorizer
    except:
        print("Training new model...")
        data = pd.read_csv('spam.csv', encoding='latin-1')
        data = data[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})

        vectorizer = CountVectorizer(stop_words='english')
        X = vectorizer.fit_transform(data['text'])
        model = MultinomialNB()
        model.fit(X, data['label'])

        joblib.dump(model, 'spam_model.joblib')
        joblib.dump(vectorizer, 'vectorizer.joblib')
        return model, vectorizer

# ====================== 🎨 CUSTOM THEME ======================
custom_theme = gr.themes.Default(
    primary_hue="indigo",
    secondary_hue="blue",
    font=[gr.themes.GoogleFont("Poppins"), "ui-sans-serif", "system-ui"]
).set(
    button_primary_background_fill="linear-gradient(90deg, #4f46e5, #7c3aed)",
    button_primary_text_color="white"
)

# ====================== ✨ PREDICTION FUNCTION ======================
def predict(text):
    text_vec = vectorizer.transform([text])
    proba = model.predict_proba(text_vec)[0]

    plot_df = pd.DataFrame({
        'class': ['HAM', 'SPAM'],
        'probability': proba,
        'color': ['#4ade80', '#f87171']  # Green for HAM, Red for SPAM
    })

    return {
        "SPAM": float(proba[1]),
        "HAM": float(proba[0])
    }, plot_df

# ====================== 🚀 LOAD MODEL ======================
model, vectorizer = load_model()

# ====================== 🖥️ INTERFACE ======================
with gr.Blocks(theme=custom_theme, css="footer {visibility: hidden}") as demo:
    # Header
    gr.Markdown("""
    <div style='text-align: center; margin-bottom: 20px'>
        <h1 style='font-weight: 800; color: #4f46e5'>🔍 SpamGuard Pro</h1>
        <p style='color: #6b7280'>AI-powered spam detection with 98.2% accuracy</p>
    </div>
    """)

    # Main Content
    with gr.Row():
        with gr.Column(scale=2):
            input_box = gr.Textbox(
                label="Message Content",
                placeholder="Paste suspicious message here...",
                lines=5
            )

            with gr.Accordion("💡 Try these examples", open=False):
                gr.Examples(
                    examples=[
                        ["Congratulations! You've won a $1000 Walmart gift card!"],
                        ["Meeting reminder: Tomorrow 10AM in Conference Room B"],
                        ["URGENT: Your bank account needs verification"],
                        ["Hey mom, can you pick up milk on your way home?"]
                    ],
                    inputs=input_box
                )

            submit_btn = gr.Button("Analyze Message", size="lg")

        with gr.Column(scale=1):
            output_label = gr.Label(
                label="Analysis Result",
                num_top_classes=2
            )

            bar_plot = gr.BarPlot(
                label="Confidence Level",
                height=200,
                width=100,
                x="class",
                y="probability",
                color="color",
                tooltip=["class", "probability"],
                container=False
            )

    # Footer
    gr.Markdown("""
    <div style='text-align: center; margin-top: 20px; color: #9ca3af'>
        <small>Powered by Naive Bayes | Model v2.1</small>
    </div>
    """)

    # Interaction
    submit_btn.click(
        fn=predict,
        inputs=input_box,
        outputs=[output_label, bar_plot]
    )

# ====================== 🎉 LAUNCH APP ======================
if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71733e22f4dc81af98.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
